# 01. Preparação de Dados, Engenharia de Atributos e Divisão Temporal

Este notebook realiza o pipeline inicial de processamento dos rankings semanais de álbuns do Spotify:
1. Carregamento dos dados brutos consolidados dos 5 países.
2. Construção da variável-alvo temporal (`target_trend_up`).
3. Extração e engenharia de variáveis explicativas sem data leakage.
4. Particionamento temporal estrito em D0 (Histórico), D1 (Recente) e D2 (Futuro).
5. Persistência física das partições na pasta `data/processed/`.

In [1]:
import sys
import os
import glob
import pandas as pd

# Permite que o notebook enxergue a pasta src/
sys.path.append(os.path.abspath('..'))
from src.data_processing import engineer_features_and_target, split_temporal_data

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

## 1. Carregamento dos Dados Brutos

Lemos o arquivo CSV consolidado contendo os registros de Brasil, Grã-Bretanha, Japão, Estados Unidos e África do Sul.

In [2]:
# Procura pelo CSV presente na pasta data/raw
arquivos = glob.glob(f"{RAW_DIR}/*.csv")
print("Arquivos encontrados:", arquivos)

# Carrega o arquivo consolidado
df_raw = pd.read_csv(arquivos[0])
print(f"Dataset bruto carregado: {df_raw.shape[0]:,} linhas e {df_raw.shape[1]} colunas.")
df_raw.head(3)

Arquivos encontrados: ['../data/raw\\spotify_albums_all_countries.csv']
Dataset bruto carregado: 281,588 linhas e 17 colunas.


,date,country,rank,uri,album_name,artist_names,label,peak_rank,previous_rank,weeks_on_chart,consecutive_weeks,entry_status,peak_date,entry_rank,entry_date,release_date,artist_uris
0,2020-11-19,br,1,spotify:album:1GAEkzl2H1mXS5byOsrc5f,Conquistas (Ao Vivo),Os Barões Da Pisadinha,Sony Music Entertainment,1,-1,1,1,NEW_ENTRY,2020-11-19,1,2020-11-19,2020-10-16,spotify:artist:5Lv2GUVwqmQBPwrTrxucE5
1,2020-11-19,br,2,spotify:album:0SV8dxSGFR0KzVzskmii6y,Ao Vivo no Ibirapuera,Henrique & Juliano,WorkShow,2,-1,1,1,NEW_ENTRY,2020-11-19,2,2020-11-19,2020-05-15,spotify:artist:3p7PcrEHaaKLJnPUGOtRlT
2,2020-11-19,br,3,spotify:album:1Yg3QTRv9IGAWeJQb7q8bT,Por Mais Beijos Ao Vivo,Zé Neto & Cristiano,Som Livre,3,-1,1,1,NEW_ENTRY,2020-11-19,3,2020-11-19,2020-03-26,spotify:artist:487N2T9nIPEHrlTZLL3SQs


## 2. Engenharia de Atributos e Definição do Target

Aplicamos a função `engineer_features_and_target`:
* **Variável-alvo (`target_trend_up`):** Compara o rank da linha atual com o rank da mesma obra na semana seguinte (`shift(-1)` agrupado por `country` e `uri`). Se a colocação for menor numericamente, o valor atribuído é 1 (subiu de posição), caso contrário 0.
* **Features construídas:** idade do álbum (`album_age_days`), distância do pico (`dist_from_peak`), variação semanal (`rank_jump_prev`), permanência e indicador de Top 50.

In [3]:
df_proc = engineer_features_and_target(df_raw)
print(f"Dataset processado: {df_proc.shape[0]:,} registros válidos.")
df_proc[['country', 'album_name', 'rank', 'target_trend_up', 'album_age_days', 'dist_from_peak']].head(5)

Dataset processado: 272,779 registros válidos.


,country,album_name,rank,target_trend_up,album_age_days,dist_from_peak
0,us,Blonde,50,1,1524,0
1,us,reputation,154,0,1077,0
2,us,Tickets To My Downfall (SOLD OUT Deluxe),5,0,23,1
3,us,Certified Hitmaker (AVA Leak),45,0,69,2
4,us,Luv Is Rage 2 (Deluxe),39,1,1070,1


## 3. Particionamento Cronológico (D0, D1, D2)

Seguindo o edital (~60% histórico, ~20% recente, ~20% futuro), realizamos os cortes cronológicos:
* **D0 (Histórico):** Registros até 29/02/2024.
* **D1 (Recente):** Registros de 01/03/2024 até 30/04/2025.
* **D2 (Futuro):** Registros a partir de 01/05/2025 até 2026.

In [4]:
d0, d1, d2 = split_temporal_data(df_proc)
total = len(df_proc)

print(f"D0 (Histórico): {len(d0):,} linhas ({len(d0)/total:.1%}) | {d0['date'].min().date()} a {d0['date'].max().date()}")
print(f"D1 (Recente)  : {len(d1):,} linhas ({len(d1)/total:.1%}) | {d1['date'].min().date()} a {d1['date'].max().date()}")
print(f"D2 (Futuro)   : {len(d2):,} linhas ({len(d2)/total:.1%}) | {d2['date'].min().date()} a {d2['date'].max().date()}")

D0 (Histórico): 163,932 linhas (60.1%) | 2020-10-22 a 2024-02-29
D1 (Recente)  : 57,497 linhas (21.1%) | 2024-03-07 a 2025-04-24
D2 (Futuro)   : 51,350 linhas (18.8%) | 2025-05-01 a 2026-05-21


## 4. Gravação Física dos Dados

Exportamos as bases de forma independente para isolar os ambientes de experimentação e garantir que D2 permaneça intocado.

In [5]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

d0.to_csv(os.path.join(PROCESSED_DIR, "d0.csv"), index=False)
d1.to_csv(os.path.join(PROCESSED_DIR, "d1.csv"), index=False)
d2.to_csv(os.path.join(PROCESSED_DIR, "d2.csv"), index=False)

print("Partições salvas com sucesso em data/processed/!")

Partições salvas com sucesso em data/processed/!
